In [22]:
import pandas as pd

In [23]:
df = pd.read_csv("data.csv", encoding='ISO-8859-1')

In [24]:
df = df.dropna(subset=['CustomerID'])

In [25]:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]


In [26]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [27]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [28]:
today_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [29]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (today_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})

In [30]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [31]:
customer_country = df.groupby('CustomerID')['Country'].first().reset_index()

rfm = rfm.merge(customer_country, on='CustomerID', how='left')

In [32]:
print(rfm.head())

   CustomerID  Recency  Frequency  Monetary         Country
0     12346.0      326          1  77183.60  United Kingdom
1     12347.0        2          7   4310.00         Iceland
2     12348.0       75          4   1797.24         Finland
3     12349.0       19          1   1757.55           Italy
4     12350.0      310          1    334.40          Norway


In [33]:
print(rfm.shape)

(4338, 5)


In [34]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

In [35]:
def segment(row):
    if row['R_score'] == 5 and row['F_score'] >= 4:
        return "Champions"
    elif row['F_score'] >= 4:
        return "Loyal Customers"
    elif row['R_score'] <= 2:
        return "At Risk"
    else:
        return "Others"

rfm['Segment'] = rfm.apply(segment, axis=1)

In [37]:
print(rfm.head())

   CustomerID  Recency  Frequency  Monetary         Country R_score F_score  \
0     12346.0      326          1  77183.60  United Kingdom       1       1   
1     12347.0        2          7   4310.00         Iceland       5       5   
2     12348.0       75          4   1797.24         Finland       2       4   
3     12349.0       19          1   1757.55           Italy       4       1   
4     12350.0      310          1    334.40          Norway       1       1   

  M_score          Segment  
0       5          At Risk  
1       5        Champions  
2       4  Loyal Customers  
3       4           Others  
4       2          At Risk  


In [39]:
rfm.to_csv("customer.csv", index=False)